# AI Workforce Capacity Planning Platform
## Notebook 07 — Enterprise Metadata Management

**Platform release:** v3.0.0  
**Implementation:** 08 — Enterprise Metadata Management  
**Release remediation:** 28 — Enterprise Release Validation  
**Canonical namespace:** `src.*`

This notebook validates the enterprise metadata subsystem, including:

1. metadata domain models,
2. Spark dataset profiling,
3. deterministic dataset fingerprinting,
4. catalog persistence and retrieval,
5. metadata service orchestration,
6. an end-to-end metadata registration workflow.

### Release Validation Scope

This notebook verifies that the metadata management workflow remains
operational under the canonical `src.*` package namespace established for
the v3.0.0 enterprise release.

In [0]:
%run ./00_project_setup

## Section 01 — Metadata Domain Validation

In [0]:
# =============================================================================
# Section 01 — Metadata Domain Validation
# Platform release: v3.0.0
# Implementation: 08
# Release remediation: 28
# =============================================================================

from src.metadata import (
    ColumnProfile,
    DatasetFingerprint,
    DatasetProfile,
    DatasetStatistics,
    MetadataCatalogEntry,
)

assert ColumnProfile is not None
assert DatasetFingerprint is not None
assert DatasetProfile is not None
assert DatasetStatistics is not None
assert MetadataCatalogEntry is not None

print("=" * 72)
print("NOTEBOOK 07 — METADATA DOMAIN VALIDATION")
print("=" * 72)
print("Platform release      : v3.0.0")
print("Canonical namespace   : src.*")
print("Implementation        : 08")
print("Release remediation   : 28")
print("Metadata domain       : PASSED")
print("=" * 72)

## Section 02 — Spark Dataset Profiler Validation

This section validates the reusable Spark profiling engine using a controlled
dataset with known row counts, duplicate rows, null values, numeric fields,
string fields, and a date field.

In [0]:
from pyspark.sql import functions as F

from src.metadata import SparkDatasetProfiler


profiler_test_df = (
    spark.createDataFrame(
        [
            (1, "Monday", 10000.0, "2026-07-27"),
            (2, "Tuesday", 10500.0, "2026-07-28"),
            (3, "Wednesday", None, "2026-07-29"),
            (3, "Wednesday", None, "2026-07-29"),
        ],
        [
            "record_id",
            "day_name",
            "released_lines",
            "release_date_text",
        ],
    )
    .withColumn(
        "release_date",
        F.to_date("release_date_text"),
    )
    .drop("release_date_text")
)

metadata_profiler = SparkDatasetProfiler(
    approximate_distinct=False
)

dataset_statistics, column_profiles = metadata_profiler.profile(
    profiler_test_df
)

print("Row count:", dataset_statistics.row_count)
print("Column count:", dataset_statistics.column_count)
print("Duplicate rows:", dataset_statistics.duplicate_row_count)
print("Null cells:", dataset_statistics.null_cell_count)
print("Columns profiled:", len(column_profiles))

assert dataset_statistics.row_count == 4
assert dataset_statistics.column_count == 4
assert dataset_statistics.duplicate_row_count == 1
assert dataset_statistics.null_cell_count == 2
assert len(column_profiles) == 4

print("=" * 72)
print("SPARK DATASET PROFILER VALIDATION")
print("=" * 72)
print("Platform release      : v3.0.0")
print("Canonical namespace   : src.*")
print("Implementation        : 08")
print("Release remediation   : 28")
print(f"Row count             : {dataset_statistics.row_count:,}")
print(f"Column count          : {dataset_statistics.column_count}")
print(f"Duplicate rows        : {dataset_statistics.duplicate_row_count}")
print(f"Null cells            : {dataset_statistics.null_cell_count}")
print(f"Columns profiled      : {len(column_profiles)}")
print("Profiler status       : PASSED")
print("=" * 72)

## Section 03 — Dataset Fingerprint Validation 

Validate deterministic schema, content, metadata, and combined dataset  
fingerprints using the profiler test dataset.

In [0]:
from src.metadata import (
    DatasetFingerprint,
    DatasetFingerprintGenerator,
)


fingerprint_generator = DatasetFingerprintGenerator()

fingerprint_metadata = {
    "dataset_key": "metadata_profiler_test",
    "dataset_layer": "TEST",
    "project_name": PROJECT_NAME,
    "environment": ENVIRONMENT,
}


first_fingerprint = fingerprint_generator.generate(
    profiler_test_df,
    metadata=fingerprint_metadata,
    statistics=dataset_statistics,
)

second_fingerprint = fingerprint_generator.generate(
    profiler_test_df,
    metadata=fingerprint_metadata,
    statistics=dataset_statistics,
)


print("Schema hash        :", first_fingerprint.schema_hash)
print("Content hash       :", first_fingerprint.content_hash)
print("Metadata hash      :", first_fingerprint.metadata_hash)
print("Combined hash      :", first_fingerprint.combined_hash)
print("Row count          :", first_fingerprint.row_count)
print("Column count       :", first_fingerprint.column_count)
print("Fingerprint version:", first_fingerprint.fingerprint_version)
print("Algorithm          :", first_fingerprint.algorithm)
print("Generated at UTC   :", first_fingerprint.generated_at_utc)


assert isinstance(
    first_fingerprint,
    DatasetFingerprint,
)

assert first_fingerprint.row_count == 4
assert first_fingerprint.column_count == 4

assert len(first_fingerprint.schema_hash) == 64
assert len(first_fingerprint.content_hash) == 64
assert len(first_fingerprint.metadata_hash) == 64
assert len(first_fingerprint.combined_hash) == 64

assert first_fingerprint.fingerprint_version == "1.0.0"
assert first_fingerprint.algorithm == "SHA-256"

assert (
    first_fingerprint.schema_hash
    == second_fingerprint.schema_hash
)

assert (
    first_fingerprint.content_hash
    == second_fingerprint.content_hash
)

assert (
    first_fingerprint.metadata_hash
    == second_fingerprint.metadata_hash
)

assert (
    first_fingerprint.combined_hash
    == second_fingerprint.combined_hash
)

assert DatasetFingerprintGenerator.fingerprints_match(
    first_fingerprint,
    second_fingerprint,
)

assert (
    first_fingerprint.generated_at_utc
    <= second_fingerprint.generated_at_utc
)

print("=" * 72)
print("DATASET FINGERPRINT VALIDATION")
print("=" * 72)
print("Platform release      : v3.0.0")
print("Canonical namespace   : src.*")
print("Implementation        : 08")
print("Release remediation   : 28")
print(f"Schema hash           : {first_fingerprint.schema_hash}")
print(f"Content hash          : {first_fingerprint.content_hash}")
print(f"Metadata hash         : {first_fingerprint.metadata_hash}")
print(f"Combined hash         : {first_fingerprint.combined_hash}")
print(f"Row count             : {first_fingerprint.row_count:,}")
print(f"Column count          : {first_fingerprint.column_count}")
print(
    f"Fingerprint version   : "
    f"{first_fingerprint.fingerprint_version}"
)
print(f"Algorithm             : {first_fingerprint.algorithm}")
print("Deterministic match   : PASSED")
print("Fingerprint status    : PASSED")
print("=" * 72)

## Section 04 — End-to-End Metadata Workflow

### 4.1 Create Controlled Sample Dataset

Create a deterministic sample dataset used to validate the complete  
enterprise metadata registration workflow.

In [0]:
sample_df = spark.createDataFrame(
    [
        (1001, "Alice", "Engineering", 85000.0),
        (1002, "Bob", "Operations", 72000.0),
        (1003, "Charlie", "Finance", 91000.0),
        (1004, "Diana", "Engineering", 88000.0),
    ],
    [
        "employee_id",
        "employee_name",
        "department",
        "salary",
    ],
)

display(sample_df)

### 4.2 Initialize Metadata Service

Initialize the canonical metadata service against the governed metadata  
catalog configured by the shared project bootstrap.

In [0]:
# =============================================================================
# 4.2 Initialize Metadata Service
# Platform release: v3.0.0
# Implementation: 08
# Release remediation: 28
# =============================================================================

from src.metadata.service import MetadataService


METADATA_CATALOG_PATH = (
    f"{METADATA_ROOT}/catalog"
)

metadata_service = MetadataService(
    spark=spark,
    catalog_path=METADATA_CATALOG_PATH,
)

print("=" * 72)
print("METADATA SERVICE INITIALIZATION")
print("=" * 72)
print("Platform release      : v3.0.0")
print("Canonical namespace   : src.*")
print("Implementation        : 08")
print("Release remediation   : 28")
print(f"Metadata catalog path : {METADATA_CATALOG_PATH}")
print("Metadata service      : INITIALIZED")
print("=" * 72)

### 4.3 Register Dataset

Profile, fingerprint, persist, and register the controlled dataset through  

the enterprise metadata service.

In [0]:
# =============================================================================
# 4.3 Register Dataset
# AI Workforce Capacity Planning Platform
# Platform release: v3.0.0
# Implementation: 08 — Enterprise Metadata Management
# Release remediation: 28 — Enterprise Release Validation
# Canonical namespace: src.*
# =============================================================================

import uuid


# -----------------------------------------------------------------------------
# Controlled dataset storage
# -----------------------------------------------------------------------------

SAMPLE_DATASET_PATH = f"{METADATA_ROOT}/sample_dataset"

execution_id = str(uuid.uuid4())


# -----------------------------------------------------------------------------
# Register dataset through the enterprise metadata service
# -----------------------------------------------------------------------------

dataset_profile, catalog_entry = metadata_service.register_dataset(
    dataframe=sample_df,
    dataset_name="Employee Sample Dataset",
    dataset_key="employee_sample_dataset",
    layer="silver",
    storage_path=SAMPLE_DATASET_PATH,
    storage_format="delta",
    execution_id=execution_id,
    pipeline_name="enterprise-metadata-demo",
    pipeline_version="1.0.0",
    owner="Data Engineering",
    business_description=(
        "Sample dataset demonstrating enterprise metadata registration."
    ),
    overwrite=True,
)


# -----------------------------------------------------------------------------
# Registration contract
# -----------------------------------------------------------------------------

assert dataset_profile is not None
assert catalog_entry is not None

assert dataset_profile.dataset_name == "Employee Sample Dataset"
assert dataset_profile.dataset_key == "employee_sample_dataset"
assert dataset_profile.layer == "silver"
assert dataset_profile.storage_path == SAMPLE_DATASET_PATH

assert catalog_entry.dataset_name == "Employee Sample Dataset"
assert catalog_entry.dataset_key == "employee_sample_dataset"
assert catalog_entry.layer == "silver"
assert catalog_entry.storage_path == SAMPLE_DATASET_PATH


# -----------------------------------------------------------------------------
# Release-validation summary
# -----------------------------------------------------------------------------

print("=" * 72)
print("METADATA DATASET REGISTRATION")
print("=" * 72)
print("Platform release      : v3.0.0")
print("Canonical namespace   : src.*")
print("Implementation        : 08")
print("Release remediation   : 28")
print(f"Execution ID          : {execution_id}")
print(f"Dataset name          : {dataset_profile.dataset_name}")
print(f"Dataset key           : {dataset_profile.dataset_key}")
print(f"Dataset layer         : {dataset_profile.layer}")
print(f"Storage path          : {dataset_profile.storage_path}")
print("DatasetProfile        : CREATED")
print("MetadataCatalogEntry  : CREATED")
print("Registration status   : PASSED")
print("=" * 72)

### 4.4 Dataset Profile

Inspect the `DatasetProfile` returned by the metadata registration workflow.

In [0]:
print(dataset_profile)

### 4.5 Metadata Catalog Entry

Inspect the persisted `MetadataCatalogEntry` generated for the registered
dataset.

In [0]:
print(catalog_entry)

### 4.6 Verify Registration

Verify that the registered dataset can be retrieved from the metadata catalog
and that catalog state is internally consistent.

In [0]:
# =============================================================================
# 4.6 Verify Registration
# Platform release: v3.0.0
# Implementation: 08
# Release remediation: 28
# =============================================================================

registered_dataset_exists = metadata_service.dataset_exists(
    "employee_sample_dataset"
)

registered_dataset = metadata_service.get_dataset(
    "employee_sample_dataset"
)

catalog_size = metadata_service.count_datasets()

assert registered_dataset_exists is True
assert registered_dataset is not None
assert registered_dataset.dataset_key == "employee_sample_dataset"
assert registered_dataset.dataset_name == "Employee Sample Dataset"
assert registered_dataset.layer == "silver"
assert registered_dataset.row_count == 4
assert registered_dataset.column_count == 4
assert catalog_size >= 1

print("=" * 72)
print("METADATA REGISTRATION VERIFICATION")
print("=" * 72)
print("Platform release      : v3.0.0")
print("Canonical namespace   : src.*")
print("Implementation        : 08")
print("Release remediation   : 28")
print(f"Dataset exists        : {registered_dataset_exists}")
print(f"Dataset key           : {registered_dataset.dataset_key}")
print(f"Dataset layer         : {registered_dataset.layer}")
print(f"Row count             : {registered_dataset.row_count:,}")
print(f"Column count          : {registered_dataset.column_count}")
print(f"Catalog size          : {catalog_size}")
print("Registration status   : PASSED")
print("=" * 72)

### 4.7 Metadata Catalog

Inspect the governed metadata catalog as a Spark DataFrame.

In [0]:
catalog_df = metadata_service.catalog_dataframe()

display(catalog_df)


## Section 05 — Execution Summary

In [0]:
# ============================================================================
# Section 05 — Execution Summary
# Platform release: v3.0.0
# Release remediation: 28
# ============================================================================

# ---------------------------------------------------------------------------
# Final contract assertions
# ---------------------------------------------------------------------------

assert dataset_profile is not None
assert catalog_entry is not None
assert registered_dataset is not None
assert catalog_df is not None

assert registered_dataset_exists is True

assert dataset_profile.dataset_key == "employee_sample_dataset"
assert catalog_entry.dataset_key == "employee_sample_dataset"
assert registered_dataset.dataset_key == "employee_sample_dataset"

assert dataset_profile.layer == "silver"
assert catalog_entry.layer == "silver"
assert registered_dataset.layer == "silver"

assert dataset_profile.storage_path == SAMPLE_DATASET_PATH
assert catalog_entry.storage_path == SAMPLE_DATASET_PATH
assert registered_dataset.storage_path == SAMPLE_DATASET_PATH

assert catalog_entry.row_count == 4
assert catalog_entry.column_count == 4

assert registered_dataset.row_count == 4
assert registered_dataset.column_count == 4

assert catalog_size >= 1
assert catalog_df.count() >= 1

registered_catalog_rows = (
    catalog_df
    .filter(catalog_df.dataset_key == "employee_sample_dataset")
    .count()
)

assert registered_catalog_rows == 1


# ---------------------------------------------------------------------------
# Release-validation summary
# ---------------------------------------------------------------------------

print("=" * 80)
print("ENTERPRISE METADATA MANAGEMENT — FINAL ACCEPTANCE")
print("=" * 80)

print("Platform release       : v3.0.0")
print("Canonical namespace    : src.*")
print("Implementation         : 08")
print("Release remediation    : 28")

print("-" * 80)

print("Dataset registration   : PASSED")
print("Dataset profiling      : PASSED")
print("Dataset fingerprinting : PASSED")
print("Metadata catalog       : PASSED")
print("Metadata service       : PASSED")
print("Registration retrieval : PASSED")
print("Catalog DataFrame      : PASSED")
print("End-to-end workflow    : PASSED")

print("-" * 80)

print(f"Dataset name           : {registered_dataset.dataset_name}")
print(f"Dataset key            : {registered_dataset.dataset_key}")
print(f"Dataset layer          : {registered_dataset.layer}")
print(f"Storage path           : {registered_dataset.storage_path}")
print(f"Storage format         : {registered_dataset.storage_format}")
print(f"Row count              : {registered_dataset.row_count:,}")
print(f"Column count           : {registered_dataset.column_count:,}")
print(f"Catalog size           : {catalog_size:,}")
print(f"Registered catalog rows: {registered_catalog_rows:,}")

print("-" * 80)

print("Metadata domain status : PASSED")
print("ENG-001 remediation    : PASSED")
print("Notebook status        : PASSED")

print("=" * 80)
print("IMPLEMENTATION 08 — ENTERPRISE METADATA MANAGEMENT COMPLETED SUCCESSFULLY")
print("=" * 80)